# MuJoCo Robot Zoo — Interactive Demo

This notebook demonstrates how to launch physics simulations for robots from the
[MuJoCo Menagerie](https://github.com/google-deepmind/mujoco_menagerie) collection — a
curated library of high-quality robot models for the
[MuJoCo](https://mujoco.org/) physics engine.

**What does this notebook do:**
1. Open a virtual desktop (VNC) where the 3D viewer will appear.
2. Automatically discover every robot scene in `../mujoco_menagerie`.
3. Click a button to instantly launch any robot in an interactive MuJoCo viewer.

> **Tip — multiple scenes per robot:**  
> Some robots ship with several scene variants (e.g. `scene.xml`, `scene_mjx.xml`, `scene_arm.xml`).  
> Each variant gets its own button so you can try them all.

## Step 1 — Open the VNC Desktop

The MuJoCo viewer is a native OpenGL window.  
Run the cell below to embed the virtual desktop right here in JupyterLab.  
If it does not show up, open it in a new browser tab.  
The simulation window will appear there once you launch a robot in Step 5.

In [ ]:
from utils import display_desktop
display_desktop()

## Step 2 — Import Libraries

| Library | Purpose |
|---|---|
| `mujoco` | Physics engine & MJCF model loader |
| `mujoco.viewer` | Passive (non-blocking) 3-D viewer |
| `threading` | Run the simulation loop in the background |
| `ipywidgets` | Interactive buttons and layout |
| `glob` | Discover XML scene files on disk |

In [ ]:
import os
import time
import threading
import glob as glob_module

import numpy as np
import mujoco
import mujoco.viewer
import ipywidgets as widgets
from IPython.display import display

print(f"MuJoCo version: {mujoco.__version__}")

## Step 3 — Simulation Engine

The `run_sim` function is the physics loop that runs inside a background thread:

1. **Load model** — `MjModel.from_xml_path()` parses the MJCF scene file.
2. **Create data** — `MjData` holds the mutable simulation state (positions, velocities, forces …).
3. **Passive viewer** — `launch_passive` opens the viewer without blocking Python; the notebook stays responsive.
4. **Random controls** — each step samples random actuator commands within the actuator's `ctrlrange`  
   so the robot moves even without a policy.  Robots with no actuators (e.g. passive fixtures) are handled gracefully.
5. **Real-time pacing** — the loop sleeps for exactly `model.opt.timestep` seconds between steps.

`launch_sim` wraps `run_sim`: it stops any already-running simulation before starting the new one,
so clicking a second robot automatically closes the first viewer.

In [ ]:
# ── Global simulation state ──────────────────────────────────────────────────
_sim_stop_event: threading.Event = threading.Event()
_sim_thread: threading.Thread | None = None


def run_sim(xml_path: str, stop_event: threading.Event) -> None:
    """Load an MJCF scene and run a real-time physics loop in a passive viewer."""
    m = mujoco.MjModel.from_xml_path(xml_path)
    d = mujoco.MjData(m)

    with mujoco.viewer.launch_passive(m, d, show_left_ui=False) as viewer:
        # Disable shadows and reflections for better performance
        viewer.user_scn.flags[mujoco.mjtRndFlag.mjRND_SHADOW] = 0
        viewer.user_scn.flags[mujoco.mjtRndFlag.mjRND_REFLECTION] = 0

        start = time.time()
        # Run for up to 10 minutes, or until the viewer is closed / stop is requested
        while viewer.is_running() and not stop_event.is_set() and time.time() - start < 600:
            step_start = time.time()

            # Apply random actuator commands (safe for models with no actuators)
            # if m.nu > 0:  # nu = number of actuators
            #     ctrl_range = m.actuator_ctrlrange
            #     d.ctrl[:] = np.random.uniform(
            #         low=ctrl_range[:, 0],
            #         high=ctrl_range[:, 1]
            #     )

            mujoco.mj_step(m, d)
            viewer.sync()

            # Real-time pacing: sleep for the remainder of the timestep
            elapsed = time.time() - step_start
            remaining = m.opt.timestep - elapsed
            if remaining > 0:
                time.sleep(remaining)


def launch_sim(xml_path: str) -> None:
    """Stop any running simulation and start a new one for *xml_path*."""
    global _sim_stop_event, _sim_thread

    # Signal the previous thread to stop and wait briefly for it to exit
    _sim_stop_event.set()
    if _sim_thread is not None and _sim_thread.is_alive():
        _sim_thread.join(timeout=2.0)

    # Start fresh
    _sim_stop_event = threading.Event()
    _sim_thread = threading.Thread(
        target=run_sim,
        args=(xml_path, _sim_stop_event),
        daemon=True,
    )
    _sim_thread.start()


print("Simulation engine ready.")

## Step 4 — Discover Robot Scenes

The cell below scans every sub-directory of `../mujoco_menagerie` and collects:

- All `scene*.xml` files (the MJCF scene descriptions).
- The first `.png` / `.jpg` image found in that directory (used as the button thumbnail).

Directories that contain no `scene*.xml` files (documentation, assets, tests …) are skipped.

In [ ]:
MENAGERIE_DIR = os.path.normpath(os.path.join(os.getcwd(), "..", "mujoco_menagerie"))

robots: list[dict] = []

for entry in sorted(os.listdir(MENAGERIE_DIR)):
    robot_path = os.path.join(MENAGERIE_DIR, entry)
    if not os.path.isdir(robot_path):
        continue

    # Discover scene XML files
    scene_files = sorted(glob_module.glob(os.path.join(robot_path, "scene*.xml")))
    if not scene_files:
        continue  # skip non-robot directories

    # Find the first image (prefer .png over .jpg)
    images = (glob_module.glob(os.path.join(robot_path, "*.png")) +
              glob_module.glob(os.path.join(robot_path, "*.jpg")) +
              glob_module.glob(os.path.join(robot_path, "*.jpeg")))
    img_path = images[0] if images else None

    robots.append({
        "name": entry,
        "path": robot_path,
        "scenes": scene_files,
        "image": img_path,
    })

print(f"Found {len(robots)} robots:\n")
for r in robots:
    scene_names = ", ".join(os.path.basename(s) for s in r["scenes"])
    img_status = "📷" if r["image"] else "  "
    print(f"  {img_status} {r['name']:35s}  scenes: {scene_names}")

## Step 5 — Interactive Robot Launcher

Run the cell below to display the robot gallery.

- Each card shows the **robot thumbnail** (if available) and one **button per scene variant**.
- Click any button to load and launch that robot in the VNC viewer (Step 1).
- Launching a new robot automatically **closes the previous viewer** — no need to restart the kernel.

> **Controls inside the viewer:**  
> `Left-drag` — rotate · `Right-drag` — pan · `Scroll` — zoom · `Ctrl+A` — reset camera  
> Press **Esc** or close the window to stop the current simulation.

In [ ]:
CARD_WIDTH  = 180   # px — width of each robot card
IMG_HEIGHT  = 100   # px — thumbnail height
COLS_PER_ROW = 4    # number of cards per row

# ── Status bar ──────────────────────────────────────────────────────────────
status_bar = widgets.HTML(
    value="<div style='padding:6px 10px;background:#e3f2fd;border-radius:4px;font-size:13px;'>"
          "<b>👆 Click a button below to launch a simulation in the VNC desktop.</b></div>"
)


def make_card(robot: dict) -> widgets.VBox:
    """Build one robot card: thumbnail + one launch button per scene."""
    children: list = []

    # ── Thumbnail ────────────────────────────────────────────────────────────
    if robot["image"]:
        with open(robot["image"], "rb") as fh:
            img_bytes = fh.read()
        ext = os.path.splitext(robot["image"])[1].lstrip(".")
        if ext in ("jpg", "jpeg"):
            ext = "jpeg"
        img_widget = widgets.Image(
            value=img_bytes,
            format=ext,
            layout=widgets.Layout(
                width=f"{CARD_WIDTH - 16}px",
                height=f"{IMG_HEIGHT}px",
                object_fit="contain",
            ),
        )
        children.append(img_widget)
    else:
        # Placeholder when no image is available
        placeholder = widgets.HTML(
            value=f"<div style='width:{CARD_WIDTH-16}px;height:{IMG_HEIGHT}px;"
                  "background:#f5f5f5;display:flex;align-items:center;"
                  "justify-content:center;font-size:28px;'>🤖</div>"
        )
        children.append(placeholder)

    # ── Robot name label ─────────────────────────────────────────────────────
    label = widgets.HTML(
        value=f"<div style='text-align:center;"
              f"word-break:break-word;max-width:{CARD_WIDTH-16}px;"
              f"color:#333;'>{robot['name']}</div>"
    )
    children.append(label)

    # ── One button per scene variant ─────────────────────────────────────────
    for scene_path in robot["scenes"]:
        scene_name = os.path.basename(scene_path).replace(".xml", "")
        btn = widgets.Button(
            description=scene_name,
            tooltip=f"Launch {robot['name']} — {scene_name}",
            layout=widgets.Layout(
                width=f"{CARD_WIDTH - 16}px",
                height="26px",
                margin="1px 0",
            ),
            button_style='primary',
        )

        def _on_click(b, path=scene_path, rname=robot["name"], sname=scene_name):
            status_bar.value = (
                f"<div style='padding:6px 10px;background:#fff9c4;border-radius:4px;font-size:13px;'>"
                f"⏳ Launching <b>{rname}</b> — <code>{sname}</code> …</div>"
            )
            launch_sim(path)
            status_bar.value = (
                f"<div style='padding:6px 10px;background:#e8f5e9;border-radius:4px;font-size:13px;'>"
                f"✅ Running <b>{rname}</b> — <code>{sname}</code>. "
                f"Switch to the VNC Desktop tab to see the viewer.</div>"
            )

        btn.on_click(_on_click)
        children.append(btn)

    return widgets.VBox(
        children,
        layout=widgets.Layout(
            border="1px solid #e0e0e0",
            border_radius="6px",
            padding="8px",
            margin="4px",
            width=f"{CARD_WIDTH}px",
            align_items="center",
            background_color="#fafafa",
        ),
    )


# ── Build gallery grid ───────────────────────────────────────────────────────
cards = [make_card(r) for r in robots[:]]

rows = []
for i in range(0, len(cards), COLS_PER_ROW):
    rows.append(
        widgets.HBox(
            cards[i : i + COLS_PER_ROW],
            layout=widgets.Layout(flex_wrap="wrap"),
        )
    )

gallery = widgets.VBox(
    [status_bar] + rows,
    layout=widgets.Layout(width="100%"),
)

display(gallery)